## Fase 1 — Inspecció i presentació del conjunt de dades

**Sector:** manteniment i reparació d'automòbils a Catalunya (Barcelona, Lleida, Tarragona, Sabadell i Figueres).

**Objectiu:** revisar la qualitat dels registres, netejar-los amb regles justificades i analitzar les hores de treball i els quilòmetres per servei, tipus de vehicle i taller.

**Columnes:** `servei`, `tipus_vehicle` i `taller` són camps descriptius; `hores_treball` és un valor decimal d'hores; `km_vehicle` és un quilometratge enter; `data` és la data del servei i ha de tenir el format `AAAA-MM-DD` (per exemple, `2004-04-13`).

No imputarem hores, quilòmetres ni dates absents: són dades necessàries per a l'anàlisi i no s'han d'inventar.

In [2]:
import csv
from datetime import datetime

# Nom del fitxer CSV corresponent als tallers mecànics
nom_fitxer = "04_taller_mecanic.csv"

# Lectura del fitxer CSV amb DictReader
with open(nom_fitxer, mode="r", encoding="utf-8", newline="") as fitxer:
    lector = csv.DictReader(fitxer)
    camps = lector.fieldnames
    registres = list(lector)

print("Camps del CSV de Tallers:", camps)
print("Nombre total de registres:", len(registres))
print("Primer registre:", registres[0] if registres else "No hi ha registres")

print("Primeres cinc files:")
for reg in registres[:5]:
    print(reg)

print(f"Dimensions: {len(registres)} files i {len(camps)} columnes")
print("Tipus originals: DictReader llegeix inicialment tots els camps com a text (str).")

Camps del CSV de Tallers: ['\ufeffservei', 'tipus_vehicle', 'taller', 'hores_treball', 'km_vehicle', 'data']
Nombre total de registres: 120
Exemple de primer registre: {'\ufeffservei': 'Pastilles de fre', 'tipus_vehicle': 'Vehicle híbrid', 'taller': 'Lleida', 'hores_treball': '220.3', 'km_vehicle': '14246', 'data': '2026-02-09'}


## Fase 2 — Detecció d'incidències i regles de qualitat

Comprovarem camps obligatoris buits, valors numèrics mal formats o fora de rang, dates invàlides i espais sobrants. Les hores accepten coma o punt decimal; els quilòmetres han de ser enters no negatius; la data ha de seguir exactament el format `AAAA-MM-DD` i existir al calendari.

In [ ]:
def converteix_hores(valor):
    """Retorna hores decimals positives o None si el valor no és vàlid."""
    import re
    text = str(valor).strip()
    if not re.fullmatch(r"\d+(?:[.,]\d+)?", text):
        return None
    try:
        hores = float(text.replace(",", "."))
        return hores if hores > 0 else None
    except (ValueError, TypeError):
        return None


def converteix_km(valor):
    """Retorna quilòmetres enters >= 0 o None si no són vàlids."""
    import re
    text = str(valor).strip()
    if not re.fullmatch(r"\d+", text):
        return None
    try:
        return int(text)
    except (ValueError, TypeError):
        return None


def data_valida(valor):
    """Comprova una data real escrita exactament com AAAA-MM-DD."""
    import re
    text = str(valor).strip()
    if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", text):
        return False
    try:
        datetime.strptime(text, "%Y-%m-%d")
        return True
    except (ValueError, TypeError):
        return False

### Neteja amb traçabilitat

Classificarem cada fila com a vàlida o rebutjada. Les correccions (com eliminar espais o convertir la coma decimal) i els motius de rebuig quedaran registrats a `incidencies`.

In [5]:
registres_nets = []
rebutjats = []
incidencies = []

camps_obligatoris = ["servei", "tipus_vehicle", "taller"]

for num_fila, reg in enumerate(registres, start=2):
    valid = True
    reg_net = {}

    # Comprovació dels camps obligatoris i eliminació d'espais
    for camp in camps_obligatoris:
        original = reg.get(camp, "")
        net = str(original).strip()

        if net == "":
            valid = False
            incidencies.append({
                "fila_csv": num_fila,
                "tipus": "REBUTJAT",
                "camp": camp,
                "valor_original": original,
                "accio": "Camp obligatori buit"
            })
        else:
            if str(original) != net:
                incidencies.append({
                    "fila_csv": num_fila,
                    "tipus": "CORRECCIO",
                    "camp": camp,
                    "valor_original": original,
                    "accio": "Espais sobrants eliminats"
                })
            reg_net[camp] = net

    # Validació de les hores
    hores_originals = reg.get("hores_treball", "")
    hores = converteix_hores(hores_originals)

    if hores is None:
        valid = False
        incidencies.append({
            "fila_csv": num_fila,
            "tipus": "REBUTJAT",
            "camp": "hores_treball",
            "valor_original": hores_originals,
            "accio": "Valor no numèric o no positiu"
        })
    else:
        if "," in str(hores_originals):
            incidencies.append({
                "fila_csv": num_fila,
                "tipus": "CORRECCIO",
                "camp": "hores_treball",
                "valor_original": hores_originals,
                "accio": "Coma decimal convertida"
            })
        reg_net["hores_treball"] = hores

    # Validació dels quilòmetres
    km_originals = reg.get("km_vehicle", "")
    km = converteix_km(km_originals)

    if km is None:
        valid = False
        incidencies.append({
            "fila_csv": num_fila,
            "tipus": "REBUTJAT",
            "camp": "km_vehicle",
            "valor_original": km_originals,
            "accio": "Valor no enter o quilometratge negatiu"
        })
    else:
        reg_net["km_vehicle"] = km

    # Validació de la data
    data_original = str(reg.get("data", "")).strip()

    if not data_valida(data_original):
        valid = False
        incidencies.append({
            "fila_csv": num_fila,
            "tipus": "REBUTJAT",
            "camp": "data",
            "valor_original": reg.get("data", ""),
            "accio": "Data buida, amb format incorrecte o no vàlida"
        })
    else:
        reg_net["data"] = data_original

    if valid:
        registres_nets.append(reg_net)
    else:
        rebutjats.append(reg)

print(f"Registres llegits: {len(registres)}")
print(f"Registres vàlids: {len(registres_nets)}")
print(f"Registres rebutjats: {len(rebutjats)}")
print(f"Incidències documentades: {len(incidencies)}")

Registres llegits: 120
Registres vàlids: 0
Registres rebutjats: 120
Incidències documentades: 176


## Fase 3 — Gestió dels registres incorrectes

Els registres rebutjats es mantenen separats i no entren als càlculs perquè els camps necessaris són incomplets o no es poden interpretar amb seguretat.

In [6]:
print("Exemples de registres rebutjats:")
for reg in rebutjats[:5]:
    print(reg)

print("\nIncidències detectades:")
for inc in incidencies[:10]:
    print(inc)

Exemples de registres rebutjats:
{'\ufeffservei': 'Pastilles de fre', 'tipus_vehicle': 'Vehicle híbrid', 'taller': 'Lleida', 'hores_treball': '220.3', 'km_vehicle': '14246', 'data': '2026-02-09'}
{'\ufeffservei': 'Diagnosi electrònica', 'tipus_vehicle': 'Turisme', 'taller': 'Tarragona', 'hores_treball': '105.6', 'km_vehicle': 'aprox. 25', 'data': '31/04/2025'}
{'\ufeffservei': 'Pastilles de fre', 'tipus_vehicle': 'Vehicle elèctric', 'taller': 'Barcelona', 'hores_treball': '171,8', 'km_vehicle': '216348', 'data': '2025-08-05'}
{'\ufeffservei': '', 'tipus_vehicle': 'Vehicle elèctric', 'taller': 'Barcelona', 'hores_treball': '206.7', 'km_vehicle': '257273', 'data': '2025-07-20'}
{'\ufeffservei': 'Diagnosi electrònica', 'tipus_vehicle': '', 'taller': '', 'hores_treball': '161.4', 'km_vehicle': '183625', 'data': '2025-03-15'}

Incidències detectades:
{'fila_csv': 2, 'tipus': 'REBUTJAT', 'camp': 'servei', 'valor_original': '', 'accio': 'Camp obligatori buit'}
{'fila_csv': 3, 'tipus': 'REBUTJ

## Fase 4 — Anàlisi de les dades netes

Calculem hores totals i mitjanes, quilometratge mitjà i agrupacions per servei, vehicle i taller.

In [7]:
if registres_nets:
    total_hores = round(sum(r["hores_treball"] for r in registres_nets), 2)
    mitjana_hores = round(total_hores / len(registres_nets), 2)
    mitjana_km = round(
        sum(r["km_vehicle"] for r in registres_nets) / len(registres_nets), 2
    )

    print(f"Serveis vàlids analitzats: {len(registres_nets)}")
    print(f"Total d'hores de treball: {total_hores}")
    print(f"Mitjana d'hores per servei: {mitjana_hores}")
    print(f"Mitjana de quilòmetres dels vehicles: {mitjana_km}")
else:
    print("No hi ha registres vàlids per analitzar.")

No hi ha registres vàlids per analitzar.


## Fase 5 — Control de qualitat

Comprovem que tots els registres llegits s'han classificat i que les dades netes compleixen les regles definides.

In [8]:
hores_per_servei = {}
hores_per_vehicle = {}
serveis_per_taller = {}

for reg in registres_nets:
    servei = reg["servei"]
    vehicle = reg["tipus_vehicle"]
    taller = reg["taller"]

    hores_per_servei[servei] = hores_per_servei.get(servei, 0) + reg["hores_treball"]
    hores_per_vehicle[vehicle] = hores_per_vehicle.get(vehicle, 0) + reg["hores_treball"]
    serveis_per_taller[taller] = serveis_per_taller.get(taller, 0) + 1

hores_per_servei = {
    k: round(v, 2) for k, v in hores_per_servei.items()
}
hores_per_vehicle = {
    k: round(v, 2) for k, v in hores_per_vehicle.items()
}

print("Hores acumulades per servei:", hores_per_servei)
print("Hores acumulades per tipus de vehicle:", hores_per_vehicle)
print("Nombre de serveis per taller:", serveis_per_taller)

assert len(registres_nets) + len(rebutjats) == len(registres)

assert all(r["hores_treball"] > 0 and r["km_vehicle"] >= 0 for r in registres_nets)

assert all(data_valida(r["data"]) for r in registres_nets)

print("Controls de qualitat superats.")


Hores acumulades per servei: {}
Hores acumulades per tipus de vehicle: {}
Nombre de serveis per taller: {}


## Fase 6 — Resultats i conclusions



La cel·la següent resumeix els resultats a partir dels registres vàlids. Interpreta els resultats tenint en compte també la proporció de registres rebutjats.


In [ ]:
percentatge_valids = round(len(registres_nets) / len(registres) * 100, 1) if registres else 0

hores_top = max(hores_per_vehicle.items(), key=lambda x: x[1]) if hores_per_vehicle else ("Cap", 0)

print(f"Registres vàlids: {len(registres_nets)} de {len(registres)} ({percentatge_valids}%)")

print(f"Registres rebutjats: {len(rebutjats)}")

print(f"Total d\x27hores de treball: {total_hores if registres_nets else 0} h")

print(f"Mitjana d\x27hores per servei vàlid: {mitjana_hores if registres_nets else 0} h")

print(f"Mitjana de quilòmetres: {mitjana_km if registres_nets else 0} km")

print(f"Tipus de vehicle amb més hores acumulades: {hores_top[0]} ({round(hores_top[1], 2)} h)")

if incidencies:

    resum_motius = {}

    for inc in incidencies:

        clau = inc["accio"]

        resum_motius[clau] = resum_motius.get(clau, 0) + 1

    print("Resum d\x27incidències per motiu:", resum_motius)
